# Análisis exploratorio de datos bursátiles

Este cuaderno realiza un análisis exploratorio de un conjunto de datos bursátiles formado por 26 empresas. Para cada *ticker* se dispone de cinco variables: precio de apertura (`open`), máximo diario (`high`), mínimo diario (`low`), precio de cierre (`close`) y volumen negociado (`volume`).

Las columnas presentan una estructura jerárquica de dos niveles: el primero identifica el *ticker* y el segundo la variable observada. Todas las variables son de tipo `float`.

El análisis se centra en describir la estructura de los datos, comprobar su consistencia, estudiar la evolución de los precios y del volumen, calcular rentabilidades, analizar la volatilidad, cuantificar las relaciones entre empresas y detectar observaciones extremas. Los resultados obtenidos tienen carácter descriptivo y se refieren únicamente al periodo contenido en el conjunto de datos.

## 1. Importación de bibliotecas y configuración

Se cargan las bibliotecas necesarias y se definen los *tickers* y atributos esperados. La configuración gráfica se mantiene sencilla para facilitar la lectura de las figuras.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import os
import pyarrow.parquet as pq

df = pq.read_table(os.path.join('..', 'DATOS', 'TRANSFORMADOS', 'df_diario.parquet')).to_pandas()

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 10

tickers = [
    'AAPL', 'AMGN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'DOW',
    'GS', 'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK',
    'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'V'
]

atributos = ['close', 'high', 'low', 'open', 'volume']

## 2. Estructura general del conjunto de datos

La primera revisión permite conocer el número de observaciones, el número de columnas, el intervalo temporal disponible y la organización interna del `DataFrame`. También se comprueba que las columnas correspondan a un `MultiIndex` con los niveles `Ticker` y `Atributo`.

In [ ]:
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")
print(f"Tipo del índice: {type(df.index).__name__}")
print(f"Tipo de las columnas: {type(df.columns).__name__}")

print("\nPrimeras observaciones:")
display(df.head())

print("\nÚltimas observaciones:")
display(df.tail())

In [ ]:
if not isinstance(df.columns, pd.MultiIndex) or df.columns.nlevels != 2:
    raise ValueError(
        "Las columnas deben ser un MultiIndex de dos niveles: (Ticker, Atributo)."
    )

df.columns = df.columns.set_names(["Ticker", "Atributo"])

tickers_presentes = df.columns.get_level_values("Ticker").unique().tolist()
atributos_presentes = df.columns.get_level_values("Atributo").unique().tolist()

print("Tickers presentes:")
print(tickers_presentes)

print("\nAtributos presentes:")
print(atributos_presentes)

print("\nTickers esperados que no aparecen:")
print(sorted(set(tickers) - set(tickers_presentes)))

print("\nTickers no incluidos en la lista inicial:")
print(sorted(set(tickers_presentes) - set(tickers)))

print("\nAtributos esperados que no aparecen:")
print(sorted(set(atributos) - set(atributos_presentes)))

print("\nAtributos adicionales:")
print(sorted(set(atributos_presentes) - set(atributos)))

In [ ]:
tipos = df.dtypes.rename("dtype").reset_index()
display(tipos)

print("\nFrecuencia de tipos de datos:")
display(tipos["dtype"].value_counts())

## 3. Revisión del índice temporal

En una serie bursátil el orden cronológico es necesario para calcular variaciones, rentabilidades y ventanas móviles. Se comprueba si el índice está ordenado y si contiene fechas duplicadas. Cuando el índice no está ordenado, se ordena de forma creciente antes de continuar.

In [ ]:
print("Índice ordenado de forma creciente:", df.index.is_monotonic_increasing)
print("Valores duplicados en el índice:", int(df.index.duplicated().sum()))

if not df.index.is_monotonic_increasing:
    df = df.sort_index()
    print("El DataFrame se ha ordenado cronológicamente.")

print("\nPrimera fecha:", df.index.min())
print("Última fecha:", df.index.max())
print("Número de observaciones temporales:", len(df))

## 4. Comprobaciones de calidad

Se revisan valores ausentes, valores infinitos y filas duplicadas. Aunque el conjunto de datos haya sido limpiado previamente, estas comprobaciones permiten documentar su estado antes de comenzar el análisis.

In [ ]:
nulos = df.isna().sum()

resumen_nulos = pd.DataFrame({
    "n_nulos": nulos,
    "porcentaje_nulos": 100 * nulos / len(df)
})

print("Número total de valores ausentes:", int(nulos.sum()))
display(resumen_nulos[resumen_nulos["n_nulos"] > 0])

print("\nNúmero de filas completamente duplicadas:", int(df.duplicated().sum()))
print("Número de valores infinitos:", int(np.isinf(df.to_numpy()).sum()))

## 5. Separación de las variables principales

Para simplificar los cálculos posteriores se extraen cinco `DataFrame`, uno por variable. Cada columna pasa a representar una empresa.

In [ ]:
close = df.xs("close", axis=1, level="Atributo").copy()
high = df.xs("high", axis=1, level="Atributo").copy()
low = df.xs("low", axis=1, level="Atributo").copy()
open_ = df.xs("open", axis=1, level="Atributo").copy()
volume = df.xs("volume", axis=1, level="Atributo").copy()

orden = [ticker for ticker in tickers if ticker in close.columns]

close = close[orden]
high = high[orden]
low = low[orden]
open_ = open_[orden]
volume = volume[orden]

print("Dimensiones de cada bloque:")
print("close :", close.shape)
print("high  :", high.shape)
print("low   :", low.shape)
print("open  :", open_.shape)
print("volume:", volume.shape)

## 6. Coherencia de las variables OHLC y del volumen

Se comprueba la relación lógica entre los precios de apertura, máximo, mínimo y cierre. El máximo diario debería ser mayor o igual que el resto de precios de la sesión, y el mínimo debería ser menor o igual. También se comprueba que no existan volúmenes negativos.

In [ ]:
checks_ohlc = pd.DataFrame(index=orden)

checks_ohlc["high < low"] = (high < low).sum()
checks_ohlc["high < open"] = (high < open_).sum()
checks_ohlc["high < close"] = (high < close).sum()
checks_ohlc["low > open"] = (low > open_).sum()
checks_ohlc["low > close"] = (low > close).sum()
checks_ohlc["volume < 0"] = (volume < 0).sum()

display(checks_ohlc)

print(
    "Número total de inconsistencias detectadas:",
    int(checks_ohlc.to_numpy().sum())
)

## 7. Estadística descriptiva de precios de cierre

Las estadísticas descriptivas permiten comparar la escala y dispersión de los precios de cierre de cada empresa. Estas magnitudes se interpretan dentro del periodo estudiado y no deben confundirse con medidas de rentabilidad.

In [ ]:
resumen_close = close.describe().T
resumen_close.index.name = "Ticker"
display(resumen_close)

## 8. Estadística descriptiva del volumen

El volumen negociado presenta habitualmente una escala muy diferente entre empresas. El resumen permite localizar series con mayor actividad y observar la dispersión de esta variable.

In [ ]:
resumen_volume = volume.describe().T
resumen_volume.index.name = "Ticker"
display(resumen_volume)

## 9. Evolución de los precios de cierre

La evolución conjunta de los precios permite observar tendencias, periodos de crecimiento o caída y diferencias de escala entre empresas.

In [ ]:
ax = close.plot(figsize=(16, 8), linewidth=1)
ax.set_title("Evolución de los precios de cierre")
ax.set_xlabel("Fecha")
ax.set_ylabel("Precio de cierre")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 10. Precios normalizados

La comparación de precios absolutos puede resultar poco clara cuando cada acción cotiza en una escala distinta. Para facilitar la comparación se normaliza cada serie tomando como base 100 su primera observación. Un valor de 120 indica que el precio se sitúa un 20 % por encima del valor inicial del periodo.

In [ ]:
close_normalizado = close.divide(close.iloc[0]).multiply(100)

ax = close_normalizado.plot(figsize=(16, 8), linewidth=1)
ax.set_title("Evolución normalizada de los precios de cierre (base 100)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Índice base 100")
ax.axhline(100, linewidth=1)
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 11. Cálculo de rentabilidades

Se calculan las rentabilidades simples y logarítmicas a partir de los precios de cierre. Las rentabilidades permiten comparar empresas con precios de cotización distintos y constituyen una base más adecuada para estudiar dispersión, correlaciones y riesgo.

In [ ]:
returns = close.pct_change(fill_method=None)
log_returns = np.log(close / close.shift(1))

print("Rentabilidades simples:")
display(returns.head())

print("\nRentabilidades logarítmicas:")
display(log_returns.head())

## 12. Estadística descriptiva de las rentabilidades

Además de media, desviación típica y cuantiles, se calculan asimetría y curtosis. Estas dos medidas permiten describir desviaciones respecto de una distribución simétrica y detectar la presencia de colas pronunciadas.

In [ ]:
resumen_returns = returns.describe().T
resumen_returns["asimetria"] = returns.skew()
resumen_returns["curtosis"] = returns.kurtosis()
resumen_returns.index.name = "Ticker"

display(resumen_returns)

## 13. Distribución de las rentabilidades diarias

Los histogramas muestran la forma de la distribución de las rentabilidades de cada empresa. Las observaciones alejadas de la zona central permiten apreciar la presencia de movimientos diarios poco frecuentes.

In [ ]:
for ticker in orden:
    datos = returns[ticker].dropna()

    plt.figure(figsize=(10, 5))
    plt.hist(datos, bins=40, alpha=0.85)
    plt.axvline(datos.mean(), linestyle="--", linewidth=1)
    plt.title(f"Distribución de las rentabilidades diarias - {ticker}")
    plt.xlabel("Rentabilidad diaria")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()

## 14. Comparación de las rentabilidades mediante diagramas de caja

El diagrama de caja permite comparar la dispersión de las rentabilidades y localizar visualmente observaciones extremas. La escala común facilita detectar qué empresas presentan movimientos diarios más amplios.

In [ ]:
plt.figure(figsize=(16, 7))
returns.boxplot(rot=90, showfliers=True)
plt.title("Distribución de las rentabilidades diarias por empresa")
plt.xlabel("Ticker")
plt.ylabel("Rentabilidad diaria")
plt.tight_layout()
plt.show()

## 15. Rentabilidad acumulada

La rentabilidad acumulada representa la evolución de una inversión teórica realizada al comienzo del periodo. El cálculo no incorpora dividendos, comisiones, impuestos ni otros costes de transacción.

In [ ]:
rentabilidad_acumulada = (1 + returns.fillna(0)).cumprod() - 1

ax = rentabilidad_acumulada.plot(figsize=(16, 8), linewidth=1)
ax.set_title("Rentabilidad acumulada")
ax.set_xlabel("Fecha")
ax.set_ylabel("Rentabilidad acumulada")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
rentabilidad_total = (close.iloc[-1] / close.iloc[0] - 1).sort_values(ascending=False)

tabla_rentabilidad_total = pd.DataFrame({
    "rentabilidad_total": rentabilidad_total,
    "rentabilidad_total_%": rentabilidad_total * 100
})

display(tabla_rentabilidad_total)

## 16. Volatilidad

La volatilidad se aproxima mediante la desviación típica de las rentabilidades. Para datos diarios se presenta también una estimación anualizada utilizando 252 sesiones bursátiles al año.

In [ ]:
media_diaria = returns.mean()
volatilidad_diaria = returns.std()

rentabilidad_media_anualizada = media_diaria * 252
volatilidad_anualizada = volatilidad_diaria * np.sqrt(252)

tabla_riesgo = pd.DataFrame({
    "rentabilidad_media_diaria": media_diaria,
    "volatilidad_diaria": volatilidad_diaria,
    "rentabilidad_media_anualizada_aprox": rentabilidad_media_anualizada,
    "volatilidad_anualizada": volatilidad_anualizada
}).sort_values("volatilidad_anualizada")

display(tabla_riesgo)

In [ ]:
volatilidad_anualizada.sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Volatilidad anualizada por empresa")
plt.xlabel("Ticker")
plt.ylabel("Volatilidad anualizada")
plt.tight_layout()
plt.show()

## 17. Relación entre rentabilidad media y volatilidad

El siguiente gráfico sitúa cada empresa según su rentabilidad media anualizada aproximada y su volatilidad anualizada. Su finalidad es descriptiva: permite comparar el comportamiento histórico observado durante el periodo disponible.

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(volatilidad_anualizada, rentabilidad_media_anualizada, s=45)

for ticker in orden:
    x = volatilidad_anualizada[ticker]
    y = rentabilidad_media_anualizada[ticker]
    plt.annotate(ticker, (x, y), xytext=(4, 4), textcoords="offset points")

plt.axhline(0, linewidth=1)
plt.xlabel("Volatilidad anualizada")
plt.ylabel("Rentabilidad media anualizada aproximada")
plt.title("Rentabilidad media y volatilidad")
plt.tight_layout()
plt.show()

## 18. Correlación entre las rentabilidades

La correlación se calcula sobre rentabilidades y no sobre precios. De esta forma se analiza hasta qué punto los movimientos diarios de dos acciones tienden a producirse en la misma dirección. Valores próximos a 1 indican una asociación lineal positiva elevada, mientras que valores próximos a 0 reflejan una relación lineal débil.

In [ ]:
corr_returns = returns.corr()
display(corr_returns.round(3))

In [ ]:
plt.figure(figsize=(13, 11))
imagen = plt.imshow(corr_returns, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(imagen, label="Correlación")
plt.xticks(range(len(corr_returns.columns)), corr_returns.columns, rotation=90)
plt.yticks(range(len(corr_returns.index)), corr_returns.index)
plt.title("Matriz de correlación de las rentabilidades diarias")
plt.tight_layout()
plt.show()

In [ ]:
pares = (
    corr_returns.where(
        np.triu(np.ones(corr_returns.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(ascending=False)
)

print("Diez pares con mayor correlación:")
display(pares.head(10).to_frame("correlacion"))

print("\nDiez pares con menor correlación:")
display(pares.tail(10).to_frame("correlacion"))

## 19. Rango intradía

El rango intradía se obtiene a partir de la diferencia entre el máximo y el mínimo de cada sesión. Para hacer comparables las empresas se expresa como proporción del precio mínimo diario.

In [ ]:
rango_intradía = (high - low) / low

resumen_rango = pd.DataFrame({
    "media": rango_intradía.mean(),
    "mediana": rango_intradía.median(),
    "maximo": rango_intradía.max()
}).sort_values("media", ascending=False)

display(resumen_rango)

In [ ]:
rango_intradía.mean().sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Rango intradía medio relativo")
plt.xlabel("Ticker")
plt.ylabel("(High - Low) / Low")
plt.tight_layout()
plt.show()

## 20. Rentabilidad intradía

La rentabilidad intradía mide la variación entre apertura y cierre de una misma sesión. Este cálculo separa el movimiento producido durante el horario de negociación del cambio ocurrido entre el cierre previo y la apertura siguiente.

In [ ]:
intraday_return = (close - open_) / open_

resumen_intraday = pd.DataFrame({
    "media": intraday_return.mean(),
    "desviacion": intraday_return.std(),
    "minimo": intraday_return.min(),
    "maximo": intraday_return.max()
}).sort_values("desviacion", ascending=False)

display(resumen_intraday)

## 21. Gaps de apertura

El *gap* de apertura compara el precio de apertura de una sesión con el cierre de la sesión anterior. Un valor positivo indica una apertura por encima del cierre previo; un valor negativo indica lo contrario.

In [ ]:
gap_apertura = open_ / close.shift(1) - 1

resumen_gap = pd.DataFrame({
    "media": gap_apertura.mean(),
    "desviacion": gap_apertura.std(),
    "minimo": gap_apertura.min(),
    "maximo": gap_apertura.max()
}).sort_values("desviacion", ascending=False)

display(resumen_gap)

In [ ]:
gap_apertura.abs().mean().sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Magnitud media absoluta del gap de apertura")
plt.xlabel("Ticker")
plt.ylabel("Gap absoluto medio")
plt.tight_layout()
plt.show()

## 22. Evolución del volumen negociado

El volumen se analiza mediante su evolución temporal y mediante una media móvil de 20 sesiones. La media móvil suaviza las oscilaciones diarias y facilita la identificación de periodos con actividad persistentemente elevada o reducida.

In [ ]:
volume_normalizado = volume.divide(volume.iloc[0]).multiply(100)

ax = volume_normalizado.plot(figsize=(16, 8), linewidth=0.9)
ax.set_title("Evolución normalizada del volumen (base 100)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Índice base 100")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
volume_ma20 = volume.rolling(20).mean()

for ticker in orden:
    plt.figure(figsize=(12, 5))
    plt.plot(volume[ticker], label="Volumen", linewidth=0.8)
    plt.plot(volume_ma20[ticker], label="Media móvil 20 sesiones", linewidth=1.2)
    plt.title(f"Volumen negociado - {ticker}")
    plt.xlabel("Fecha")
    plt.ylabel("Volumen")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 23. Relación entre volumen y magnitud del movimiento del precio

Se calcula la correlación entre el volumen y el valor absoluto de la rentabilidad diaria. Esta medida permite comprobar si las sesiones con mayor actividad tienden a coincidir con movimientos de precio más intensos, con independencia de que dichos movimientos sean positivos o negativos.

In [ ]:
corr_vol_mov = pd.Series(index=orden, dtype=float)

for ticker in orden:
    datos = pd.concat(
        [volume[ticker], returns[ticker].abs()],
        axis=1
    ).dropna()
    datos.columns = ["volume", "abs_return"]
    corr_vol_mov[ticker] = datos["volume"].corr(datos["abs_return"])

corr_vol_mov = corr_vol_mov.sort_values(ascending=False)

display(corr_vol_mov.to_frame("corr_volumen_abs_rentabilidad"))

In [ ]:
corr_vol_mov.sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Correlación entre volumen y magnitud de la rentabilidad diaria")
plt.xlabel("Ticker")
plt.ylabel("Correlación")
plt.axhline(0, linewidth=1)
plt.tight_layout()
plt.show()

## 24. Medias móviles de precios

Las medias móviles de 20 y 50 sesiones se utilizan para suavizar la serie de precios y observar cambios de tendencia con menor influencia del ruido diario. Se representan por separado para cada empresa para evitar superponer demasiadas series en una sola figura.

In [ ]:
ma20 = close.rolling(20).mean()
ma50 = close.rolling(50).mean()

for ticker in orden:
    plt.figure(figsize=(12, 5))
    plt.plot(close[ticker], label="Cierre", linewidth=1)
    plt.plot(ma20[ticker], label="Media móvil 20", linewidth=1)
    plt.plot(ma50[ticker], label="Media móvil 50", linewidth=1)
    plt.title(f"Precio de cierre y medias móviles - {ticker}")
    plt.xlabel("Fecha")
    plt.ylabel("Precio")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 25. Volatilidad móvil

La volatilidad no suele permanecer constante a lo largo del tiempo. Para observar su evolución se calcula una desviación típica móvil de 30 sesiones y se anualiza mediante el factor correspondiente a 252 sesiones bursátiles.

In [ ]:
rolling_vol_30 = returns.rolling(30).std() * np.sqrt(252)

ax = rolling_vol_30.plot(figsize=(16, 8), linewidth=0.9)
ax.set_title("Volatilidad móvil anualizada de 30 sesiones")
ax.set_xlabel("Fecha")
ax.set_ylabel("Volatilidad anualizada")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 26. Drawdown

El *drawdown* mide la caída de una serie respecto de su máximo histórico previo dentro del periodo analizado. Esta medida complementa la volatilidad porque refleja directamente la magnitud de las pérdidas sufridas desde un máximo anterior.

In [ ]:
maximo_acumulado = close.cummax()
drawdown = close / maximo_acumulado - 1

max_drawdown = drawdown.min().sort_values()

tabla_drawdown = pd.DataFrame({
    "max_drawdown": max_drawdown,
    "max_drawdown_%": max_drawdown * 100
})

display(tabla_drawdown)

In [ ]:
ax = drawdown.plot(figsize=(16, 8), linewidth=0.9)
ax.set_title("Drawdown a lo largo del periodo")
ax.set_xlabel("Fecha")
ax.set_ylabel("Drawdown")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
max_drawdown.sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Máximo drawdown observado por empresa")
plt.xlabel("Ticker")
plt.ylabel("Máximo drawdown")
plt.tight_layout()
plt.show()

## 27. Sesiones con mayores movimientos

Se identifican para cada empresa las cinco mayores subidas y las cinco mayores caídas diarias. Esta revisión permite localizar fechas que merecen un análisis posterior y comprobar si determinados episodios extremos afectan simultáneamente a varias empresas.

In [ ]:
extremos = {}

for ticker in orden:
    serie = returns[ticker].dropna()
    extremos[ticker] = {
        "mayores_subidas": serie.nlargest(5),
        "mayores_caidas": serie.nsmallest(5)
    }

for ticker in orden:
    print("=" * 70)
    print(ticker)
    print("\nMayores subidas:")
    display(extremos[ticker]["mayores_subidas"].to_frame("rentabilidad"))
    print("Mayores caídas:")
    display(extremos[ticker]["mayores_caidas"].to_frame("rentabilidad"))

## 28. Detección descriptiva de valores extremos mediante IQR

Como complemento se aplica la regla del rango intercuartílico a las rentabilidades. Una observación se considera extrema cuando queda por debajo de `Q1 - 1.5·IQR` o por encima de `Q3 + 1.5·IQR`. Este criterio se utiliza únicamente para describir la frecuencia de movimientos alejados del centro de la distribución; no implica que las observaciones sean erróneas.

In [ ]:
outliers_iqr = {}

for ticker in orden:
    serie = returns[ticker].dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    mascara = (serie < limite_inf) | (serie > limite_sup)

    outliers_iqr[ticker] = {
        "n_outliers": int(mascara.sum()),
        "porcentaje": 100 * mascara.mean(),
        "limite_inferior": limite_inf,
        "limite_superior": limite_sup
    }

tabla_outliers = pd.DataFrame(outliers_iqr).T.sort_values(
    "porcentaje",
    ascending=False
)

display(tabla_outliers)

In [ ]:
tabla_outliers["porcentaje"].sort_values().plot(kind="bar", figsize=(14, 6))
plt.title("Porcentaje de rentabilidades extremas según el criterio IQR")
plt.xlabel("Ticker")
plt.ylabel("Porcentaje de observaciones")
plt.tight_layout()
plt.show()

## 29. Evolución anual

Cuando el periodo contiene varios años, una agregación anual permite comparar el comportamiento de cada empresa en intervalos homogéneos. La rentabilidad anual se obtiene a partir de los precios de cierre de final de año.

In [ ]:
if isinstance(close.index, pd.DatetimeIndex):
    close_fin_ano = close.resample("YE").last()
    rentabilidad_anual = close_fin_ano.pct_change(fill_method=None)

    print("Rentabilidades por año:")
    display(rentabilidad_anual)

    plt.figure(figsize=(16, 8))
    imagen = plt.imshow(rentabilidad_anual.T, aspect="auto")
    plt.colorbar(imagen, label="Rentabilidad")
    plt.xticks(
        range(len(rentabilidad_anual.index)),
        [fecha.year for fecha in rentabilidad_anual.index],
        rotation=45
    )
    plt.yticks(range(len(rentabilidad_anual.columns)), rentabilidad_anual.columns)
    plt.title("Rentabilidades anuales por empresa")
    plt.xlabel("Año")
    plt.ylabel("Ticker")
    plt.tight_layout()
    plt.show()
else:
    print(
        "El índice no es de tipo DatetimeIndex. "
        "No se realiza la agregación anual."
    )

## 30. Tabla resumen del análisis

Se reúnen varias métricas en una única tabla para disponer de una visión compacta del comportamiento de cada empresa durante el periodo analizado. Las medidas anualizadas suponen que los datos tienen frecuencia diaria y utilizan 252 sesiones bursátiles por año.

In [ ]:
tabla_resumen = pd.DataFrame(index=orden)

tabla_resumen["precio_inicial"] = close.iloc[0]
tabla_resumen["precio_final"] = close.iloc[-1]
tabla_resumen["rentabilidad_total"] = close.iloc[-1] / close.iloc[0] - 1
tabla_resumen["rentabilidad_media_diaria"] = returns.mean()
tabla_resumen["volatilidad_diaria"] = returns.std()
tabla_resumen["rentabilidad_media_anualizada_aprox"] = returns.mean() * 252
tabla_resumen["volatilidad_anualizada"] = returns.std() * np.sqrt(252)
tabla_resumen["max_drawdown"] = drawdown.min()
tabla_resumen["rango_intradía_medio"] = rango_intradía.mean()
tabla_resumen["gap_abs_medio"] = gap_apertura.abs().mean()
tabla_resumen["volumen_medio"] = volume.mean()
tabla_resumen["corr_volumen_abs_return"] = corr_vol_mov

display(tabla_resumen)

## 31. Observaciones finales

El análisis exploratorio permite caracterizar el comportamiento histórico de las empresas desde varias perspectivas. Los precios normalizados facilitan la comparación de trayectorias; las rentabilidades permiten estudiar variación y dependencia; la volatilidad y el *drawdown* describen dimensiones distintas del riesgo observado; y las variables intradía, los *gaps* y el volumen aportan información adicional sobre la dinámica de cada acción.

Las correlaciones calculadas muestran relaciones contemporáneas dentro del periodo disponible, pero no establecen causalidad. Del mismo modo, las rentabilidades y volatilidades históricas no constituyen por sí mismas una estimación de resultados futuros.

A partir de este punto pueden plantearse análisis posteriores, como agrupación de empresas según comportamiento, reducción de dimensionalidad, estudio de regímenes de volatilidad, modelos de series temporales o construcción de variables para modelos predictivos.